In [4]:
import sympy as sp
import sympy.vector as sv
from IPython.display import display

In [5]:
uT, uB, uL, uR, uc, u_ext = sp.symbols("u_T u_B u_L u_R u_c u_ext")
uT_p, uB_p, uL_p, uR_p, uc_p, u_ext_p = sp.symbols(
    "u_T^+ u_B^+ u_L^+ u_R^+ u_c^+ u_ext^+"
)
uT_m, uB_m, uL_m, uR_m, uc_m, u_ext_m = sp.symbols(
    "u_T^- u_B^- u_L^- u_R^- u_c^- u_ext^-"
)
u = []
for i in range(3):
    u.append(
        [sp.symbols("u_{i%+d\\,j%+d}" % (i, j)) for j in range(3)]
        + [sp.symbols("u_{i%+d\\,j%+d}" % (i, j)) for j in range(-2, 0)]
    )
for i in range(-2, 0):
    u.append(
        [sp.symbols("u_{i%+d\\,j%+d}" % (i, j)) for j in range(3)]
        + [sp.symbols("u_{i%+d\\,j%+d}" % (i, j)) for j in range(-2, 0)]
    )


def flatten(u):
    return [item for sublist in u for item in sublist]


x_i, x_im1, x_ip1 = sp.symbols("x_i x_(i-1) x_(i+1)")
y_j, y_jm1, y_jp1 = sp.symbols("y_j y_(j-1) y_(j+1)")
dx, dy = sp.Symbol(r"\Delta x"), sp.Symbol(r"\Delta y")
xL, xR = sp.symbols("x_L x_R")
yT, yB = sp.symbols("y_T y_B")
# p, pL, pR, pT, pB = sp.symbols(
#     "\\mathbf{p} \\mathbf{p}_L \\mathbf{p}_R \\mathbf{p}_T \\mathbf{p}_B"
# )
x_ext, y_ext = sp.symbols("x_ext y_ext")
theta_L, theta_R, theta_T, theta_B = sp.symbols("theta_L theta_R theta_T theta_B")
coord = sv.CoordSys3D("coord")
x, y = coord.x, coord.y

nx, ny = sp.symbols("n_x n_y")
a, a_tau, b = sp.symbols("a, a_{\\tau}, b")
beta_jump, beta_p, beta_m = sp.symbols("[\\beta], beta^+, beta^-")
# nx, ny = sp.symbols("n_x n_y", cls=sp.Function)
# a, a_tau, b = sp.symbols("a, a_{\\tau}, b", cls=sp.Function)

A = sp.Matrix(
    [
        [x_i**2, x_i * yT, yT**2, x_i, yT, 1],
        [x_i**2, x_i * yB, yB**2, x_i, yB, 1],
        [xL**2, xL * y_j, y_j**2, xL, y_j, 1],
        [xR**2, xR * y_j, y_j**2, xR, y_j, 1],
        [x_i**2, x_i * y_j, y_j**2, x_i, y_j, 1],
        [x_ext**2, x_ext * y_ext, y_ext**2, x_ext, y_ext, 1],
    ]
)
A_inv = A.inv()
P_coeff = A_inv @ sp.Matrix([uT, uB, uL, uR, uc, u_ext])
A, B, C, D, E, F = P_coeff
P = A * x**2 + B * x * y + C * y**2 + D * x + E * y + F

### Case 1 at $x_R$

In [6]:
case1_right_vars_m = {
    x: x_i + theta_R * dx,
    y: y_j,
    xL: x_i - dx,
    xR: x_i + theta_R * dx,
    yT: y_j + dy,
    yB: y_j - dy,
    x_ext: x_i - dx,
    y_ext: y_j - dy,
    uc: u[0][0],
    uL: u[-1][0],
    uR: uR_m,
    uB: u[0][-1],
    uT: u[0][1],
    u_ext: u[-1][-1],
}

case1_right_vars_p = {
    x: x_i - (1 - theta_R) * dx,
    y: y_j,
    xL: x_i - (1 - theta_R) * dx,
    xR: x_i + dx,
    yT: y_j + dy,
    yB: y_j - dy,
    x_ext: x_i + dx,
    y_ext: y_j - dy,
    uc: u[1][0],
    uL: uR_p,
    uR: u[2][0],
    uB: u[1][-1],
    uT: u[1][1],
    u_ext: u[1][-1],
}

In [7]:
grad_P = sv.gradient(P)
dudx = grad_P.components[coord.i]
dudy = grad_P.components[coord.j]

In [8]:
# geometric discretization of [\beta u_x]
beta_ux_jump_geometry = (
    b * nx - beta_jump * ny * (-ny * dudx + nx * dudy) - beta_p * a_tau * ny
).subs(case1_right_vars_m)

# algebraic definition of [\beta u_x]
beta_ux_jump_algebra = beta_p * dudx.subs(case1_right_vars_p) - beta_m * dudx.subs(
    case1_right_vars_m
)

# equate the two definitions
equality = beta_ux_jump_algebra - beta_ux_jump_geometry

In [9]:
# M
eq_sub = equality.subs({uR_p: uR_m + a}).expand().collect(uR_m)
uR_m_coeff = eq_sub.coeff(uR_m).simplify()
uR_m_coeff = uR_m_coeff.collect([beta_p, beta_m, beta_jump])
tops = uR_m_coeff.as_numer_denom()[0].as_ordered_terms()
bot = uR_m_coeff.as_numer_denom()[1]
M = sp.Add(*[(top / bot).cancel().factor() for top in tops])

In [10]:
# Separate terms in 'rest' into those involving u variables and those not
u_terms = []
non_u_terms = []
rest = (eq_sub - uR_m_coeff * uR_m).simplify().expand()
for term in rest.as_ordered_terms():
    if any(u_var in term.free_symbols for sublist in u for u_var in sublist):
        u_terms.append(term)
    else:
        non_u_terms.append(term)

# rest_u_group = sp.Add(*u_terms)
# rest_non_u_group = sp.Add(*non_u_terms)

d = -sp.Add(
    *[
        term.cancel().factor()
        for term in sp.Add(*non_u_terms).collect([a, b, a_tau]).as_ordered_terms()
    ]
)
Nu = -sp.Add(
    *[
        term.cancel().factor()
        for term in sp.Add(*u_terms).collect(flatten(u)).as_ordered_terms()
    ]
)

In [28]:
# display(M)
# display(d)
# display(Nu)
print("Nu/M")
display((Nu / M).expand().collect(flatten(u)))
print("d/M")
display((d / M).simplify().collect([a, b, a_tau]))

Nu/M


-[\beta]*n_x*n_y*theta_R*u_{i-1,j-1}/(-2*[\beta]*\Delta y*n_y**2*theta_R/(\Delta x*theta_R**2 + \Delta x*theta_R) - [\beta]*\Delta y*n_y**2/(\Delta x*theta_R**2 + \Delta x*theta_R) + 2*\Delta y*beta^+*theta_R/(\Delta x*theta_R**2 - 3*\Delta x*theta_R + 2*\Delta x) - 3*\Delta y*beta^+/(\Delta x*theta_R**2 - 3*\Delta x*theta_R + 2*\Delta x) - 2*\Delta y*beta^-*theta_R/(\Delta x*theta_R**2 + \Delta x*theta_R) - \Delta y*beta^-/(\Delta x*theta_R**2 + \Delta x*theta_R)) - [\beta]*n_x*n_y*u_{i+0,j+1}/(-4*[\beta]*\Delta y*n_y**2*theta_R/(\Delta x*theta_R**2 + \Delta x*theta_R) - 2*[\beta]*\Delta y*n_y**2/(\Delta x*theta_R**2 + \Delta x*theta_R) + 4*\Delta y*beta^+*theta_R/(\Delta x*theta_R**2 - 3*\Delta x*theta_R + 2*\Delta x) - 6*\Delta y*beta^+/(\Delta x*theta_R**2 - 3*\Delta x*theta_R + 2*\Delta x) - 4*\Delta y*beta^-*theta_R/(\Delta x*theta_R**2 + \Delta x*theta_R) - 2*\Delta y*beta^-/(\Delta x*theta_R**2 + \Delta x*theta_R)) + u_{i+0,j+0}*(-[\beta]*n_x*n_y*theta_R/(-2*[\beta]*\Delta y*n_

d/M


theta_R*(theta_R + 1)*(\Delta x*(theta_R - 2)*(theta_R - 1)*(a_{\tau}*beta^+*n_y - b*n_x) + a*beta^+*(2*theta_R - 3))/(-beta^+*theta_R*(theta_R + 1)*(2*theta_R - 3) + (theta_R - 2)*(theta_R - 1)*(2*theta_R + 1)*([\beta]*n_y**2 + beta^-))

In [27]:
laplacian_x = (
    beta_m * (uR_m - u[0][0]) / (theta_R * dx)
    - beta_m * (u[0][0] - uL_m) / (theta_L * dx)
) / ((theta_L + theta_R) * dx / 2)

laplacian_y = (
    beta_m * (uT_m - u[0][0]) / (theta_T * dy)
    - beta_m * (u[0][0] - uB_m) / (theta_B * dy)
) / ((theta_T + theta_B) * dy / 2)

# f = sp.symbols("f")

laplacian = (
    (laplacian_x + laplacian_y)
    .subs(
        {
            uR_m: Nu / M + d / M,
            uL_m: u[-1][0],
            uT_m: u[0][1],
            uB_m: u[0][-1],
            theta_L: 1,
            theta_T: 1,
            theta_B: 1,
        }
    )
    .expand()
    .collect(flatten(u))
)
laplacian.coeff(u[0][0]).simplify().factor().cancel()
# sp.Add(
#     *[term.simplify() for term in laplacian.coeff(u[0][0]).as_ordered_terms()]
# )
# laplacian.as_ordered_terms()

(-4*[\beta]*\Delta x**2*beta^-*n_y**2*theta_R**3 + 10*[\beta]*\Delta x**2*beta^-*n_y**2*theta_R**2 - 2*[\beta]*\Delta x**2*beta^-*n_y**2*theta_R - 4*[\beta]*\Delta x**2*beta^-*n_y**2 + 2*[\beta]*\Delta x*\Delta y*beta^-*n_x*n_y*theta_R**3 - 6*[\beta]*\Delta x*\Delta y*beta^-*n_x*n_y*theta_R**2 + 4*[\beta]*\Delta x*\Delta y*beta^-*n_x*n_y*theta_R - 2*[\beta]*\Delta y**2*beta^-*n_y**2*theta_R**2 + 6*[\beta]*\Delta y**2*beta^-*n_y**2*theta_R - 4*[\beta]*\Delta y**2*beta^-*n_y**2 + 4*\Delta x**2*beta^+*beta^-*theta_R**3 - 2*\Delta x**2*beta^+*beta^-*theta_R**2 - 6*\Delta x**2*beta^+*beta^-*theta_R - 4*\Delta x**2*beta^-**2*theta_R**3 + 10*\Delta x**2*beta^-**2*theta_R**2 - 2*\Delta x**2*beta^-**2*theta_R - 4*\Delta x**2*beta^-**2 + 4*\Delta y**2*beta^+*beta^-*theta_R**2 - 2*\Delta y**2*beta^+*beta^-*theta_R - 6*\Delta y**2*beta^+*beta^- - 2*\Delta y**2*beta^-**2*theta_R**2 + 6*\Delta y**2*beta^-**2*theta_R - 4*\Delta y**2*beta^-**2)/(2*[\beta]*\Delta x**2*\Delta y**2*n_y**2*theta_R**3 - 5*

In [ ]:
def compute_um(vars_m: dict, vars_p: dict):
    grad_P = sv.gradient(P)
    dudx = grad_P.components[coord.i]
    dudy = grad_P.components[coord.j]

    # geometric discretization of [\beta u_x]
    beta_ux_jump_geometry = (
        b * nx - beta_jump * ny * (-ny * dudx + nx * dudy) - beta_p * a_tau * ny
    ).subs(case1_right_vars_m)

    # algebraic definition of [\beta u_x]
    beta_ux_jump_algebra = beta_p * dudx.subs(case1_right_vars_p) - beta_m * dudx.subs(
        case1_right_vars_m
    )

    # equate the two definitions
    equality = beta_ux_jump_algebra - beta_ux_jump_geometry

    # M
    eq_sub = equality.subs({uR_p: uR_m + a}).expand().collect(uR_m)
    uR_m_coeff = eq_sub.coeff(uR_m).simplify()
    uR_m_coeff = uR_m_coeff.collect([beta_p, beta_m, beta_jump])
    tops = uR_m_coeff.as_numer_denom()[0].as_ordered_terms()
    bot = uR_m_coeff.as_numer_denom()[1]
    M = sp.Add(*[(top / bot).cancel().factor() for top in tops])

    # separate terms in 'rest' into those involving u variables and those not
    u_terms = []
    non_u_terms = []
    rest = (eq_sub - uR_m_coeff * uR_m).simplify().expand()
    for term in rest.as_ordered_terms():
        if any(u_var in term.free_symbols for sublist in u for u_var in sublist):
            u_terms.append(term)
        else:
            non_u_terms.append(term)

    d = -sp.Add(
        *[
            term.cancel().factor()
            for term in sp.Add(*non_u_terms).collect([a, b, a_tau]).as_ordered_terms()
        ]
    )
    Nu = -sp.Add(
        *[
            term.cancel().factor()
            for term in sp.Add(*u_terms).collect(flatten(u)).as_ordered_terms()
        ]
    )

    Nu_M = (Nu / M).simplify().collect(flatten(u))
    d_M = (d / M).simplify().collect([a, b, a_tau])

### Case 1 at $x_L$

In [8]:
case1_left_vars_m = {
    x: x_i - theta_L * dx,
    y: y_j,
    xL: x_i - theta_L * dx,
    xR: x_i + dx,
    yT: y_j + dy,
    yB: y_j - dy,
    x_ext: x_i + dx,
    y_ext: y_j - dy,
    uc: u[0][0],
    uL: uL_m,
    uR: u[1][0],
    uB: u[0][-1],
    uT: u[0][1],
    u_ext: u[1][-1],
}

case1_left_vars_p = {
    x: x_i + (1 - theta_L) * dx,
    y: y_j,
    xL: x_i - dx,
    xR: x_i + (1 - theta_L) * dx,
    yT: y_j + dy,
    yB: y_j - dy,
    x_ext: x_i - dx,
    y_ext: y_j - dy,
    uc: u[-1][0],
    uL: u[-2][0],
    uR: uL_p,
    uB: u[-1][-1],
    uT: u[-1][1],
    u_ext: u[-2][-1],
}

In [9]:
beta_ux_jump_geometry = (
    b * nx - beta_jump * ny * (-ny * dudx + nx * dudy) - beta_p * a_tau * ny
).subs(case1_left_vars_m)

# algebraic definition of [\beta u_x]
beta_ux_jump_algebra = beta_p * dudx.subs(case1_left_vars_p) - beta_m * dudx.subs(
    case1_left_vars_m
)

# equate the two definitions
equality = beta_ux_jump_algebra - beta_ux_jump_geometry

In [10]:
# M
eq_sub = equality.subs({uL_p: uL_m + a}).expand().collect(uL_m)
uL_m_coeff = eq_sub.coeff(uL_m).simplify()
uL_m_coeff = uL_m_coeff.collect([beta_p, beta_m, beta_jump])
tops = uL_m_coeff.as_numer_denom()[0].as_ordered_terms()
bot = uL_m_coeff.as_numer_denom()[1]
print("M")
sp.Add(*[(top / bot).cancel().factor() for top in tops])

M


[\beta]*n_y**2*(2*theta_L + 1)/(\Delta x*theta_L*(theta_L + 1)) - beta^+*(2*theta_L - 3)/(\Delta x*(theta_L - 2)*(theta_L - 1)) + beta^-*(2*theta_L + 1)/(\Delta x*theta_L*(theta_L + 1))

In [11]:
u_terms = []
non_u_terms = []
rest = (eq_sub - uL_m_coeff * uL_m).simplify().expand()
for term in rest.as_ordered_terms():
    if any(u_var in term.free_symbols for sublist in u for u_var in sublist):
        u_terms.append(term)
    else:
        non_u_terms.append(term)

rest_u_group = sp.Add(*u_terms)
rest_non_u_group = sp.Add(*non_u_terms)

rest_non_u_group = sp.Add(
    *[
        term.cancel().factor()
        for term in rest_non_u_group.collect([a, b, a_tau]).as_ordered_terms()
    ]
)
rest_u_group = sp.Add(
    *[
        term.cancel().factor()
        for term in rest_u_group.collect(flatten(u)).as_ordered_terms()
    ]
)

print("-d:")
display(rest_non_u_group)
print("-Nu:")
display(rest_u_group)

-d:


a_{\tau}*beta^+*n_y - b*n_x - a*beta^+*(2*theta_L - 3)/(\Delta x*(theta_L - 2)*(theta_L - 1))

-Nu:


[\beta]*n_x*n_y*theta_L*u_{i+1,j-1}/\Delta y + [\beta]*n_x*n_y*u_{i+0,j+1}/(2*\Delta y) - [\beta]*n_x*n_y*u_{i+0,j-1}*(2*theta_L + 1)/(2*\Delta y) - beta^+*u_{i-1,j+0}*(theta_L - 2)/(\Delta x*(theta_L - 1)) + beta^+*u_{i-2,j+0}*(theta_L - 1)/(\Delta x*(theta_L - 2)) - theta_L*u_{i+1,j+0}*([\beta]*\Delta x*n_x*n_y*theta_L + [\beta]*\Delta x*n_x*n_y - [\beta]*\Delta y*n_y**2 - \Delta y*beta^-)/(\Delta x*\Delta y*(theta_L + 1)) + u_{i+0,j+0}*([\beta]*\Delta x*n_x*n_y*theta_L**2 - [\beta]*\Delta y*n_y**2*theta_L - [\beta]*\Delta y*n_y**2 - \Delta y*beta^-*theta_L - \Delta y*beta^-)/(\Delta x*\Delta y*theta_L)

### Case 2 at $x_R$ and $x_T$

In [21]:
case2_right_vars_m = {
    x: x_i + theta_R * dx,
    y: y_j,
    xL: x_i - dx,
    xR: x_i + theta_R * dx,
    yT: y_j + theta_T * dy,
    yB: y_j - dy,
    x_ext: x_i - dx,
    y_ext: y_j - dy,
    uc: u[0][0],
    uL: u[-1][0],
    uR: uR_m,
    uB: u[0][-1],
    uT: uT_m,
    u_ext: u[-1][-1],
}

case2_top_vars_m = {
    x: x_i,
    y: y_j + theta_T * dy,
    xL: x_i - dx,
    xR: x_i + theta_R * dx,
    yT: y_j + theta_T * dy,
    yB: y_j - dy,
    x_ext: x_i - dx,
    y_ext: y_j - dy,
    uc: u[0][0],
    uL: u[-1][0],
    uR: uR_m,
    uB: u[0][-1],
    uT: uT_m,
    u_ext: u[-1][-1],
}

case2_right_vars_p = {
    x: x_i - (1 - theta_R) * dx,
    y: y_j,
    xL: x_i - (1 - theta_R) * dx,
    xR: x_i + dx,
    yT: y_j + dy,
    yB: y_j - dy,
    x_ext: x_i + dx,
    y_ext: y_j - dy,
    uc: u[1][0],
    uL: uR_p,
    uR: u[2][0],
    uB: u[1][-1],
    uT: u[1][1],
    u_ext: u[1][-1],
}

case2_top_vars_p = {
    x: x_i,
    y: y_j - (1 - theta_T) * dy,
    xL: x_i - dx,
    xR: x_i + dx,
    yT: y_j + dy,
    yB: y_j - (1 - theta_T) * dy,
    x_ext: x_i + dx,
    y_ext: y_j + dy,
    uc: u[0][1],
    uL: u[-1][1],
    uR: u[1][1],
    uB: uT_p,
    uT: u[0][2],
    u_ext: u[2][2],
}

In [22]:
beta_ux_jump_geometry = (
    b * nx - beta_jump * ny * (-ny * dudx + nx * dudy) - beta_p * a_tau * ny
).subs(case2_right_vars_m)
beta_ux_jump_algebra = beta_p * dudx.subs(case2_right_vars_p) - beta_m * dudx.subs(
    case2_right_vars_m
)
equality_x = beta_ux_jump_algebra - beta_ux_jump_geometry

beta_uy_jump_geometry = (
    b * ny + beta_jump * nx * (-ny * dudx + nx * dudy) + beta_p * a_tau * nx
).subs(case2_top_vars_m)
beta_uy_jump_algebra = beta_p * dudy.subs(case2_top_vars_p) - beta_m * dudy.subs(
    case2_top_vars_m
)
equality_y = beta_uy_jump_algebra - beta_uy_jump_geometry

In [26]:
# M
M = sp.Matrix([[0, 0], [0, 0]])
Nu = sp.Matrix([0, 0])
d = sp.Matrix([0, 0])
eq_sub_x = equality_x.subs({uR_p: uR_m + a, uT_p: uT_m + a}).expand()
uR_m_coeff = eq_sub_x.coeff(uR_m).simplify()
uR_m_coeff = uR_m_coeff.collect([beta_p, beta_m, beta_jump])
tops = uR_m_coeff.as_numer_denom()[0].as_ordered_terms()
bot = uR_m_coeff.as_numer_denom()[1]
print("M[0,0]")
M[0, 0] = sp.Add(*[(top / bot).cancel().factor() for top in tops])
display(M[0, 0])

uT_m_coeff = eq_sub_x.coeff(uT_m).simplify()
uT_m_coeff = uT_m_coeff.collect([beta_p, beta_m, beta_jump])
tops = uT_m_coeff.as_numer_denom()[0].as_ordered_terms()
bot = uT_m_coeff.as_numer_denom()[1]
print("M[0,1]")
M[0, 1] = sp.Add(*[(top / bot).cancel().factor() for top in tops])
display(M[0, 1])


u_terms = []
non_u_terms = []
rest = (eq_sub_x - uR_m_coeff * uR_m - uT_m_coeff * uT_m).simplify().expand()
for term in rest.as_ordered_terms():
    if any(u_var in term.free_symbols for sublist in u for u_var in sublist):
        u_terms.append(term)
    else:
        non_u_terms.append(term)

# have a minus sign since they are on the other side of equality
# Nu[0] = -sp.Add(*u_terms)
# d[0] = -sp.Add(*non_u_terms)

d[0] = -sp.Add(
    *[
        term.cancel().factor()
        for term in sp.Add(*non_u_terms).collect([a, b, a_tau]).as_ordered_terms()
    ]
)
Nu[0] = -sp.Add(
    *[
        term.cancel().factor()
        for term in sp.Add(*u_terms).collect(flatten(u)).as_ordered_terms()
    ]
)

print("d[0]")
display(d[0])
print("Nu[0]")
display(Nu[0])

M[0,0]


-[\beta]*n_y**2*(2*theta_R + 1)/(\Delta x*theta_R*(theta_R + 1)) + beta^+*(2*theta_R - 3)/(\Delta x*(theta_R - 2)*(theta_R - 1)) - beta^-*(2*theta_R + 1)/(\Delta x*theta_R*(theta_R + 1))

M[0,1]


[\beta]*n_x*n_y/(\Delta y*theta_T*(theta_T + 1))

d[0]


-a_{\tau}*beta^+*n_y + b*n_x - a*beta^+*(2*theta_R - 3)/(\Delta x*(theta_R - 2)*(theta_R - 1))

Nu[0]


-[\beta]*n_x*n_y*theta_R*u_{i-1,j-1}/\Delta y + [\beta]*n_x*n_y*u_{i+0,j-1}*(theta_R*theta_T + theta_R + theta_T)/(\Delta y*(theta_T + 1)) - beta^+*u_{i+1,j+0}*(theta_R - 2)/(\Delta x*(theta_R - 1)) + beta^+*u_{i+2,j+0}*(theta_R - 1)/(\Delta x*(theta_R - 2)) + theta_R*u_{i-1,j+0}*([\beta]*\Delta x*n_x*n_y*theta_R + [\beta]*\Delta x*n_x*n_y + [\beta]*\Delta y*n_y**2 + \Delta y*beta^-)/(\Delta x*\Delta y*(theta_R + 1)) - u_{i+0,j+0}*([\beta]*\Delta x*n_x*n_y*theta_R**2*theta_T + [\beta]*\Delta x*n_x*n_y*theta_R*theta_T - [\beta]*\Delta x*n_x*n_y*theta_R + [\beta]*\Delta y*n_y**2*theta_R*theta_T + [\beta]*\Delta y*n_y**2*theta_T + \Delta y*beta^-*theta_R*theta_T + \Delta y*beta^-*theta_T)/(\Delta x*\Delta y*theta_R*theta_T)

In [27]:
eq_sub_y = equality_y.subs({uR_p: uR_m + a, uT_p: uT_m + a}).expand()
uR_m_coeff = eq_sub_y.coeff(uR_m).simplify()
uR_m_coeff = uR_m_coeff.collect([beta_p, beta_m, beta_jump])
tops = uR_m_coeff.as_numer_denom()[0].as_ordered_terms()
bot = uR_m_coeff.as_numer_denom()[1]
print("M[1,0]")
M[1, 0] = sp.Add(*[(top / bot).cancel().factor() for top in tops])
display(M[1, 0])

uT_m_coeff = eq_sub_y.coeff(uT_m).simplify()
uT_m_coeff = uT_m_coeff.collect([beta_p, beta_m, beta_jump])
tops = uT_m_coeff.as_numer_denom()[0].as_ordered_terms()
bot = uT_m_coeff.as_numer_denom()[1]
print("M[1,1]")
M[1, 1] = sp.Add(*[(top / bot).cancel().factor() for top in tops])
display(M[1, 1])

u_terms = []
non_u_terms = []
rest = (eq_sub_y - uR_m_coeff * uR_m - uT_m_coeff * uT_m).simplify().expand()
for term in rest.as_ordered_terms():
    if any(u_var in term.free_symbols for sublist in u for u_var in sublist):
        u_terms.append(term)
    else:
        non_u_terms.append(term)

# rest_u_group = sp.Add(*u_terms)
# rest_non_u_group = sp.Add(*non_u_terms)

d[1] = -sp.Add(
    *[
        term.cancel().factor()
        for term in sp.Add(*non_u_terms).collect([a, b, a_tau]).as_ordered_terms()
    ]
)
Nu[1] = -sp.Add(
    *[
        term.cancel().factor()
        for term in sp.Add(*u_terms).collect(flatten(u)).as_ordered_terms()
    ]
)

print("d[1]")
display(d[1])
print("Nu[1]")
display(Nu[1])

M[1,0]


[\beta]*n_x*n_y/(\Delta x*theta_R*(theta_R + 1))

M[1,1]


-[\beta]*n_x**2*(2*theta_T + 1)/(\Delta y*theta_T*(theta_T + 1)) + beta^+*(2*theta_T - 3)/(\Delta y*(theta_T - 2)*(theta_T - 1)) - beta^-*(2*theta_T + 1)/(\Delta y*theta_T*(theta_T + 1))

d[1]


a_{\tau}*beta^+*n_x + b*n_y - a*beta^+*(2*theta_T - 3)/(\Delta y*(theta_T - 2)*(theta_T - 1))

Nu[1]


-[\beta]*n_x*n_y*theta_T*u_{i-1,j-1}/\Delta x + [\beta]*n_x*n_y*u_{i-1,j+0}*(theta_R*theta_T + theta_R + theta_T)/(\Delta x*(theta_R + 1)) - beta^+*u_{i+0,j+1}*(theta_T - 2)/(\Delta y*(theta_T - 1)) + beta^+*u_{i+0,j+2}*(theta_T - 1)/(\Delta y*(theta_T - 2)) + theta_T*u_{i+0,j-1}*([\beta]*\Delta x*n_x**2 + [\beta]*\Delta y*n_x*n_y*theta_T + [\beta]*\Delta y*n_x*n_y + \Delta x*beta^-)/(\Delta x*\Delta y*(theta_T + 1)) - u_{i+0,j+0}*([\beta]*\Delta x*n_x**2*theta_R*theta_T + [\beta]*\Delta x*n_x**2*theta_R + [\beta]*\Delta y*n_x*n_y*theta_R*theta_T**2 + [\beta]*\Delta y*n_x*n_y*theta_R*theta_T - [\beta]*\Delta y*n_x*n_y*theta_T + \Delta x*beta^-*theta_R*theta_T + \Delta x*beta^-*theta_R)/(\Delta x*\Delta y*theta_R*theta_T)

In [32]:
display(M)
display(Nu)
display(d)

Matrix([
[-[\beta]*n_y**2*(2*theta_R + 1)/(\Delta x*theta_R*(theta_R + 1)) + beta^+*(2*theta_R - 3)/(\Delta x*(theta_R - 2)*(theta_R - 1)) - beta^-*(2*theta_R + 1)/(\Delta x*theta_R*(theta_R + 1)),                                                                                                                                           [\beta]*n_x*n_y/(\Delta y*theta_T*(theta_T + 1))],
[                                                                                                                                          [\beta]*n_x*n_y/(\Delta x*theta_R*(theta_R + 1)), -[\beta]*n_x**2*(2*theta_T + 1)/(\Delta y*theta_T*(theta_T + 1)) + beta^+*(2*theta_T - 3)/(\Delta y*(theta_T - 2)*(theta_T - 1)) - beta^-*(2*theta_T + 1)/(\Delta y*theta_T*(theta_T + 1))]])

Matrix([
[-[\beta]*n_x*n_y*theta_R*u_{i-1,j-1}/\Delta y + [\beta]*n_x*n_y*u_{i+0,j-1}*(theta_R*theta_T + theta_R + theta_T)/(\Delta y*(theta_T + 1)) - beta^+*u_{i+1,j+0}*(theta_R - 2)/(\Delta x*(theta_R - 1)) + beta^+*u_{i+2,j+0}*(theta_R - 1)/(\Delta x*(theta_R - 2)) + theta_R*u_{i-1,j+0}*([\beta]*\Delta x*n_x*n_y*theta_R + [\beta]*\Delta x*n_x*n_y + [\beta]*\Delta y*n_y**2 + \Delta y*beta^-)/(\Delta x*\Delta y*(theta_R + 1)) - u_{i+0,j+0}*([\beta]*\Delta x*n_x*n_y*theta_R**2*theta_T + [\beta]*\Delta x*n_x*n_y*theta_R*theta_T - [\beta]*\Delta x*n_x*n_y*theta_R + [\beta]*\Delta y*n_y**2*theta_R*theta_T + [\beta]*\Delta y*n_y**2*theta_T + \Delta y*beta^-*theta_R*theta_T + \Delta y*beta^-*theta_T)/(\Delta x*\Delta y*theta_R*theta_T)],
[-[\beta]*n_x*n_y*theta_T*u_{i-1,j-1}/\Delta x + [\beta]*n_x*n_y*u_{i-1,j+0}*(theta_R*theta_T + theta_R + theta_T)/(\Delta x*(theta_R + 1)) - beta^+*u_{i+0,j+1}*(theta_T - 2)/(\Delta y*(theta_T - 1)) + beta^+*u_{i+0,j+2}*(theta_T - 1)/(\Delta y*(theta_T - 2

Matrix([
[-a_{\tau}*beta^+*n_y + b*n_x - a*beta^+*(2*theta_R - 3)/(\Delta x*(theta_R - 2)*(theta_R - 1))],
[ a_{\tau}*beta^+*n_x + b*n_y - a*beta^+*(2*theta_T - 3)/(\Delta y*(theta_T - 2)*(theta_T - 1))]])

In [16]:
case2_left_vars_m = {
    x: x_i - theta_L * dx,
    y: y_j,
    xL: x_i - theta_L * dx,
    xR: x_i + dx,
    yT: y_j + dy,
    yB: y_j - theta_B * dy,
    x_ext: x_i + dx,
    y_ext: y_j + dy,
    uc: u[0][0],
    uL: uL_m,
    uR: u[1][0],
    uB: uB_m,
    uT: u[0][1],
    u_ext: u[1][1],
}

case2_bot_vars_m = {
    x: x_i,
    y: y_j - theta_B * dy,
    xL: x_i - theta_L * dx,
    xR: x_i + dx,
    yT: y_j + dy,
    yB: y_j - theta_B * dy,
    x_ext: x_i + dx,
    y_ext: y_j + dy,
    uc: u[0][0],
    uL: uL_m,
    uR: u[1][0],
    uB: uB_m,
    uT: u[0][1],
    u_ext: u[1][1],
}

case2_left_vars_p = {
    x: x_i + (1 - theta_L) * dx,
    y: y_j,
    xL: x_i - dx,
    xR: x_i + (1 - theta_L) * dx,
    yT: y_j + dy,
    yB: y_j - dy,
    x_ext: x_i - dx,
    y_ext: y_j - dy,
    uc: u[-1][0],
    uL: u[-2][0],
    uR: uL_p,
    uB: u[-1][-1],
    uT: u[-1][1],
    u_ext: u[-1][-1],
}

case2_bot_vars_p = {
    x: x_i,
    y: y_j + (1 - theta_B) * dy,
    xL: x_i - dx,
    xR: x_i + dx,
    yT: y_j + dy,
    yB: y_j - (1 - theta_B) * dy,
    x_ext: x_i - dx,
    y_ext: y_j - dy,
    uc: u[0][-1],
    uL: u[-1][-1],
    uR: u[1][-1],
    uB: u[1][-2],
    uT: uB_p,
    u_ext: u[-2][-2],
}

In [17]:
beta_ux_jump_geometry = (
    b * nx - beta_jump * ny * (-ny * dudx + nx * dudy) - beta_p * a_tau * ny
).subs(case2_left_vars_m)
beta_ux_jump_algebra = beta_p * dudx.subs(case2_left_vars_p) - beta_m * dudx.subs(
    case2_left_vars_m
)
equality_x = beta_ux_jump_algebra - beta_ux_jump_geometry

beta_uy_jump_geometry = (
    b * ny + beta_jump * nx * (-ny * dudx + nx * dudy) + beta_p * a_tau * nx
).subs(case2_bot_vars_m)
beta_uy_jump_algebra = beta_p * dudy.subs(case2_bot_vars_p) - beta_m * dudy.subs(
    case2_bot_vars_m
)
equality_y = beta_uy_jump_algebra - beta_uy_jump_geometry

In [18]:
# M
eq_sub_x = equality_x.subs({uL_p: uL_m + a, uB_p: uB_m + a}).expand()
uL_m_coeff = eq_sub_x.coeff(uL_m).simplify()
uL_m_coeff = uL_m_coeff.collect([beta_p, beta_m, beta_jump])
tops = uL_m_coeff.as_numer_denom()[0].as_ordered_terms()
bot = uL_m_coeff.as_numer_denom()[1]
print("M[0,0]")
display(sp.Add(*[(top / bot).cancel().factor() for top in tops]))

uB_m_coeff = eq_sub_x.coeff(uB_m).simplify()
uB_m_coeff = uB_m_coeff.collect([beta_p, beta_m, beta_jump])
tops = uB_m_coeff.as_numer_denom()[0].as_ordered_terms()
bot = uB_m_coeff.as_numer_denom()[1]
print("M[0,1]")
display(sp.Add(*[(top / bot).cancel().factor() for top in tops]))


u_terms = []
non_u_terms = []
rest = (eq_sub_x - uL_m_coeff * uL_m - uB_m_coeff * uB_m).simplify().expand()
for term in rest.as_ordered_terms():
    if any(u_var in term.free_symbols for sublist in u for u_var in sublist):
        u_terms.append(term)
    else:
        non_u_terms.append(term)

rest_u_group = sp.Add(*u_terms)
rest_non_u_group = sp.Add(*non_u_terms)

rest_non_u_group = sp.Add(
    *[
        term.cancel().factor()
        for term in rest_non_u_group.collect([a, b, a_tau]).as_ordered_terms()
    ]
)
rest_u_group = sp.Add(
    *[
        term.cancel().factor()
        for term in rest_u_group.collect(flatten(u)).as_ordered_terms()
    ]
)

print("-d[0]")
display(rest_non_u_group)
print("-Nu[0]")
display(rest_u_group)

M[0,0]


[\beta]*n_y**2*(2*theta_L + 1)/(\Delta x*theta_L*(theta_L + 1)) - beta^+*(2*theta_L - 3)/(\Delta x*(theta_L - 2)*(theta_L - 1)) + beta^-*(2*theta_L + 1)/(\Delta x*theta_L*(theta_L + 1))

M[0,1]


-[\beta]*n_x*n_y/(\Delta y*theta_B*(theta_B + 1))

-d[0]


a_{\tau}*beta^+*n_y - b*n_x - a*beta^+*(2*theta_L - 3)/(\Delta x*(theta_L - 2)*(theta_L - 1))

-Nu[0]


-[\beta]*n_x*n_y*theta_L*u_{i+1,j+1}/\Delta y + [\beta]*n_x*n_y*u_{i+0,j+1}*(theta_B*theta_L + theta_B + theta_L)/(\Delta y*(theta_B + 1)) - beta^+*u_{i-1,j+0}*(theta_L - 2)/(\Delta x*(theta_L - 1)) + beta^+*u_{i-2,j+0}*(theta_L - 1)/(\Delta x*(theta_L - 2)) + theta_L*u_{i+1,j+0}*([\beta]*\Delta x*n_x*n_y*theta_L + [\beta]*\Delta x*n_x*n_y + [\beta]*\Delta y*n_y**2 + \Delta y*beta^-)/(\Delta x*\Delta y*(theta_L + 1)) - u_{i+0,j+0}*([\beta]*\Delta x*n_x*n_y*theta_B*theta_L**2 + [\beta]*\Delta x*n_x*n_y*theta_B*theta_L - [\beta]*\Delta x*n_x*n_y*theta_L + [\beta]*\Delta y*n_y**2*theta_B*theta_L + [\beta]*\Delta y*n_y**2*theta_B + \Delta y*beta^-*theta_B*theta_L + \Delta y*beta^-*theta_B)/(\Delta x*\Delta y*theta_B*theta_L)

In [19]:
eq_sub_y = equality_y.subs({uL_p: uL_m + a, uB_p: uB_m + a}).expand()
uL_m_coeff = eq_sub_y.coeff(uL_m).simplify()
uL_m_coeff = uL_m_coeff.collect([beta_p, beta_m, beta_jump])
tops = uL_m_coeff.as_numer_denom()[0].as_ordered_terms()
bot = uL_m_coeff.as_numer_denom()[1]
print("M[1,0]")
display(sp.Add(*[(top / bot).cancel().factor() for top in tops]))

uB_m_coeff = eq_sub_y.coeff(uB_m).simplify()
uB_m_coeff = uB_m_coeff.collect([beta_p, beta_m, beta_jump])
tops = uB_m_coeff.as_numer_denom()[0].as_ordered_terms()
bot = uB_m_coeff.as_numer_denom()[1]
print("M[1,1]")
display(sp.Add(*[(top / bot).cancel().factor() for top in tops]))

u_terms = []
non_u_terms = []
rest = (eq_sub_y - uL_m_coeff * uL_m - uB_m_coeff * uB_m).simplify().expand()
for term in rest.as_ordered_terms():
    if any(u_var in term.free_symbols for sublist in u for u_var in sublist):
        u_terms.append(term)
    else:
        non_u_terms.append(term)

rest_u_group = sp.Add(*u_terms)
rest_non_u_group = sp.Add(*non_u_terms)

rest_non_u_group = sp.Add(
    *[
        term.cancel().factor()
        for term in rest_non_u_group.collect([a, b, a_tau]).as_ordered_terms()
    ]
)
rest_u_group = sp.Add(
    *[
        term.cancel().factor()
        for term in rest_u_group.collect(flatten(u)).as_ordered_terms()
    ]
)

print("-d[1]")
display(rest_non_u_group)
print("-Nu[1]")
display(rest_u_group)

M[1,0]


-[\beta]*n_x*n_y/(\Delta x*theta_L*(theta_L + 1))

M[1,1]


[\beta]*n_x**2*(2*theta_B + 1)/(\Delta y*theta_B*(theta_B + 1)) + 3*beta^+*(theta_B - 1)/(\Delta y*(theta_B - 2)) + beta^-*(2*theta_B + 1)/(\Delta y*theta_B*(theta_B + 1))

-d[1]


-a_{\tau}*beta^+*n_x - b*n_y + 3*a*beta^+*(theta_B - 1)/(\Delta y*(theta_B - 2))

-Nu[1]


-[\beta]*n_x*n_y*theta_B*u_{i+1,j+1}/\Delta x + [\beta]*n_x*n_y*u_{i+1,j+0}*(theta_B*theta_L + theta_B + theta_L)/(\Delta x*(theta_L + 1)) - beta^+*u_{i+0,j-1}*(3*theta_B - 2)/(\Delta y*(theta_B - 1)) - beta^+*u_{i+1,j-2}*(2*theta_B - 1)/(\Delta y*(theta_B - 2)*(theta_B - 1)) + theta_B*u_{i+0,j+1}*([\beta]*\Delta x*n_x**2 + [\beta]*\Delta y*n_x*n_y*theta_B + [\beta]*\Delta y*n_x*n_y + \Delta x*beta^-)/(\Delta x*\Delta y*(theta_B + 1)) - u_{i+0,j+0}*([\beta]*\Delta x*n_x**2*theta_B*theta_L + [\beta]*\Delta x*n_x**2*theta_L + [\beta]*\Delta y*n_x*n_y*theta_B**2*theta_L + [\beta]*\Delta y*n_x*n_y*theta_B*theta_L - [\beta]*\Delta y*n_x*n_y*theta_B + \Delta x*beta^-*theta_B*theta_L + \Delta x*beta^-*theta_L)/(\Delta x*\Delta y*theta_B*theta_L)